In [ ]:
import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# plt.rc("font", family="Nimbus Roman")
plt.rc("font", family="Nimbus Roman No9 L")

SMALL_SIZE = 7
MEDIUM_SIZE = 7
BIGGER_SIZE = 7

plt.rc("font", size=SMALL_SIZE)  # controls default text sizes
plt.rc("axes", titlesize=SMALL_SIZE)  # fontsize of the axes title
plt.rc("axes", labelsize=MEDIUM_SIZE)  # fontsize of the x and y labels
plt.rc("xtick", labelsize=SMALL_SIZE)  # fontsize of the tick labels
plt.rc("ytick", labelsize=SMALL_SIZE)  # fontsize of the tick labels
plt.rc("legend", fontsize=SMALL_SIZE)  # legend fontsize
plt.rc("figure", titlesize=BIGGER_SIZE)  # fontsize of the figure title

In [ ]:
sim_lst = ["m12b", "m12c", "m12f", "m12i", "m12m"]
sim_dir = "/Users/z5114326/Documents/simulations/"

all_data = sim_dir + "m12i" + "/" + "m12i" + "_res7100/snapshot_times.txt"
all_snaps = pd.read_table(all_data, comment="#", header=None, sep=r"\s+")
all_snaps.columns = [
    "index",
    "scale_factor",
    "redshift",
    "time_Gyr",
    "lookback_time_Gyr",
    "time_width_Myr",
]

tims = all_snaps["time_Gyr"].values
timez0 = tims[-1]

scmap = plt.get_cmap("tab10")
colors = [scmap(i) for i in np.linspace(0, 1, len(sim_lst))]
sim_colors = {sim: colors[i] for i, sim in enumerate(sim_lst)}

In [ ]:
cnt_dict = {sim: {"tims": tims, "avg": None, "std": None} for sim in sim_lst}

for sim in sim_lst:
    ghost_file = f"{sim_dir}{sim}/{sim}_ghosts.hdf5"
    ghost_data = h5py.File(ghost_file, "r")

    arr = []
    for it_id in ghost_data.keys():
        src_dat = ghost_data[it_id]["source"]

        tfor = src_dat["tfor"][()]
        tacc = src_dat["tacc"][()]
        tdis = src_dat["tdis"][()]
        tdis = np.where(tdis == -1, np.inf, tdis)

        cnts = np.zeros(len(tims))
        for i, tim in enumerate(tims):
            msk = (tfor <= tim) & (tacc <= tim) & (tim < tdis)
            cnts[i] = np.sum(msk)

        arr.append(cnts)

    cnt_dict[sim]["avg"] = np.mean(arr, axis=0)
    cnt_dict[sim]["std"] = np.std(arr, axis=0)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(3.5, 2))

for sim in sim_lst:
    ax.plot(timez0 - cnt_dict[sim]["tims"], cnt_dict[sim]["avg"], c=sim_colors[sim], label=sim, lw=1)

ax.set_xlim(timez0, 0)
ax.set_ylim(0, 11500)

ax.set_xlabel("Lookback Time (Gyr)")
ax.set_ylabel(r"N$_{\rm GC}$")

ax.legend()

save_loc = "../data/figures/gc_plots/gc_counts.pdf"
plt.savefig(save_loc, bbox_inches="tight")